# V035 capacity proof

Inherits frozen V029 objective tests; additionally checks full four-layer optimizer/RNG resume and memory reset. Synthetic CUDA timing gates the ordinary lease before approved training.

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,copy,time,gc
import numpy as np
import torch

V035_CAPACITY_SPEC=dict(family='lstm',hidden=320,layers=4,dropout=.15,bidirectional=True,joint_embedding_dim=0,rc_heads=False,residual=True,head_width=32,widths=[320,256,192,128],output_concat=True)

def run_capacity_preflight_v035(project_root,output_directory,device='cpu',deep_spec=None):
 P=Path(project_root).resolve();O=Path(output_directory);O.mkdir(parents=True,exist_ok=False);checks={}
 def check(name,c):
  if not bool(c):raise AssertionError(name)
  checks[name]=True
 def code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
 def equal(a,b):
  if torch.is_tensor(a):return torch.is_tensor(b) and torch.equal(a,b)
  if isinstance(a,dict):return a.keys()==b.keys() and all(equal(a[k],b[k]) for k in a)
  if isinstance(a,(list,tuple)):return len(a)==len(b) and all(equal(x,y) for x,y in zip(a,b))
  return a==b
 check('capacity_spec_exact',deep_spec==V035_CAPACITY_SPEC)
 old=P/'research/laplace_preflight_v029.ipynb';oldcode=code(old);check('inherited_proof_source_pinned',hashlib.sha256(oldcode.encode()).hexdigest()=='4143266b792b6d6eb19fa1437d755a2f5685abdfe4ccfaaeeab1a7bd7743aca3')
 ns={};exec(compile(oldcode,str(old),'exec'),ns);inherited=ns['run_laplace_preflight_v029'](P,O/'inherited',device=device,deep_spec=deep_spec);checks.update({'inherited_'+k:v for k,v in inherited['checks'].items()})
 ns={};sources={}
 for name in ['feature_components_v005.ipynb','sequence_components_v005.ipynb','training_components_v008.ipynb','laplace_components_v029.ipynb']:
  source=code(P/'research'/name);sources[name]=hashlib.sha256(source.encode()).hexdigest();exec(compile(source,name,'exec'),ns)
 selected=ns['_sequence_device'](device);ns['seed_sequence'](20261102);m=ns['v029_laplace_factory'](deep_spec,47,{}).to(selected);opt=torch.optim.Adam(m.parameters(),lr=.0015);x=torch.randn(2,80,48,device=selected);x[:,:,-1]=0;target=torch.randn(2,80,device=selected);mask=torch.ones((2,80),device=selected,dtype=torch.bool);mask[:,40:]=False
 def step(model,optimizer):
  model.train();optimizer.zero_grad(set_to_none=True);mu=model(x);loss=ns['laplace_curve_loss_v029'](mu,target,model.v029_scale,mask)[0];loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.);optimizer.step();return loss.detach()
 check('full_model_four_bidirectional_layers',deep_spec['layers']==len(deep_spec['widths'])==4 and all(deep_spec['widths'][i]>deep_spec['widths'][i+1] for i in range(3)))
 step(m,opt);state=copy.deepcopy(m.state_dict());optimizer_state=copy.deepcopy(opt.state_dict());cpu_rng=torch.get_rng_state();cuda_rng=torch.cuda.get_rng_state_all() if selected.type=='cuda' else None
 expected_loss=step(m,opt);expected=copy.deepcopy(m.state_dict());expected_optimizer=copy.deepcopy(opt.state_dict());clone=ns['v029_laplace_factory'](deep_spec,47,{}).to(selected);co=torch.optim.Adam(clone.parameters(),lr=.0015);clone.load_state_dict(state);co.load_state_dict(optimizer_state);torch.set_rng_state(cpu_rng)
 if cuda_rng is not None:torch.cuda.set_rng_state_all(cuda_rng)
 actual_loss=step(clone,co);check('actual_full_capacity_model_exact_optimizer_RNG_resume',equal(expected,clone.state_dict()) and equal(expected_optimizer,co.state_dict()) and torch.equal(expected_loss,actual_loss))
 clone.eval()
 with torch.no_grad():
  p1=clone(x);z=x.clone();z[1,:,:-1]+=10;p2=clone(z)
 check('full_capacity_memory_resets_between_breaths',torch.equal(p1[0],p2[0]));check('full_capacity_finite_outputs_and_scale',torch.isfinite(p1).all() and torch.isfinite(clone.v029_scale).all() and (clone.v029_scale>0).all())
 parameter_count=sum(p.numel() for p in clone.parameters());old_spec=dict(deep_spec,hidden=256,layers=3,widths=[256,192,128]);old_model=ns['v029_laplace_factory'](old_spec,47,{}).to(selected);old_count=sum(p.numel() for p in old_model.parameters());check('capacity_increases_only_widths_and_depth',parameter_count>old_count and {k:v for k,v in deep_spec.items() if k not in ['hidden','layers','widths']}=={k:v for k,v in old_spec.items() if k not in ['hidden','layers','widths']})
 budget=None;profiles=list(inherited['efficiency_profiles'])
 del m,opt,clone,co,old_model;gc.collect()
 if selected.type=='cuda':
  torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats();ns['seed_sequence'](20261102);old_model=ns['v029_laplace_factory'](old_spec,47,{}).to(selected);oo=torch.optim.Adam(old_model.parameters(),lr=.0015);tx=torch.randn(256,80,48,device=selected);tx[:,:,-1]=0;ty=torch.randn(256,80,device=selected);tm=torch.ones((256,80),dtype=torch.bool,device=selected);tm[:,40:]=False;times=[]
  for repeat in range(5):
   torch.cuda.synchronize();start=time.perf_counter();oo.zero_grad(set_to_none=True);mu=old_model(tx);loss=ns['laplace_curve_loss_v029'](mu,ty,old_model.v029_scale,tm)[0];loss.backward();torch.nn.utils.clip_grad_norm_(old_model.parameters(),1.);oo.step();torch.cuda.synchronize()
   if repeat>=2:times.append(time.perf_counter()-start)
  old_seconds=float(np.median(times));new_seconds=profiles[1]['median_step_seconds'];ratio=new_seconds/old_seconds
  # V029 observed 8548.893 seconds for200epochs at .16013 seconds per full step.
  # Use a conservative1.20 overhead over measured full-batch optimizer time.
  projected=200*239*new_seconds*1.20
  budget=dict(new_full_step_seconds=new_seconds,old_full_step_seconds=old_seconds,step_ratio=ratio,projected_training_seconds=projected,ordinary_lease_seconds=21600,reserve_seconds=1800,projection_overhead_multiplier=1.20)
  check('full_budget_fits_ordinary_lease_with_30minute_reserve',projected<=19800)
  check('capacity_peak_memory_below_10GiB',profiles[1]['peak_allocated_bytes']<10*1024**3)
 result=dict(status='passed',created_utc=datetime.now(timezone.utc).isoformat(),checks=checks,check_count=len(checks),source_code_hashes=sources,capacity_spec=deep_spec,parameter_count=parameter_count,old_parameter_count=old_count,actual_cuda=selected.type=='cuda',exact_resume=True,native_inference=True,lease_budget_gate_passed=True if budget else None,lease_budget=budget,efficiency_profiles=profiles,synthetic_only=True,approved_data_fits=0,raw_approved_csv_reads=0,runtime=dict(torch=str(torch.__version__),numpy=np.__version__,device=device))
 (O/'proof.json').write_text(json.dumps(result,indent=2)+'\n');print(json.dumps(result,indent=2));return result
